In [ ]:
import matplotlib_inline
from matplotlib import pyplot as plt

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
plt.style.use("math.mplstyle")

In [ ]:
import numpy as np
import netCDF4 as nc
from pyproj import CRS, Transformer
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
import rasterio
from matplotlib import pyplot as plt

ecos_df = nc.Dataset("ecostress.nc", "r")
landsat_df = nc.Dataset("landsat.nc", "r")

ecos_crs = CRS.from_epsg(32611)
ecos_x = ecos_df["xdim"][:]
ecos_y = ecos_df["ydim"][:]
ecos_shape = (len(ecos_y), len(ecos_x))

ecos_transform = from_bounds(
    west=ecos_x[0] - 35,
    south=ecos_y[-1] - 35,
    east=ecos_x[-1] + 35,
    north=ecos_y[0] + 35,
    width=len(ecos_x),
    height=len(ecos_y),
)

landsat_crs = CRS.from_proj4(
    "+proj=aea +lat_0=23 +lon_0=-96 +lat_1=29.5 +lat_2=45.5 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)
landsat_x = landsat_df["xdim"][:]
landsat_y = landsat_df["ydim"][:]
landsat_shape = (len(landsat_y), len(landsat_x))

landsat_transform = from_bounds(
    west=landsat_x[0] - 15,
    south=landsat_y[-1] - 15,
    east=landsat_x[-1] + 15,
    north=landsat_y[0] + 15,
    width=len(landsat_x),
    height=len(landsat_y),
)

print(f"ECOSTRESS grid: {ecos_shape} at 70m resolution")
print(f"Landsat grid: {landsat_shape} at 30m resolution")
print(f"ECOSTRESS CRS: {ecos_crs}")
print(f"Landsat CRS: {landsat_crs}")

In [ ]:
def compute_mean_images_at_observation_time(ecos_idx=500, landsat_idx=0):
    ecostress_fourier_coeffs_full = np.load("results.npy")
    landsat_fourier_coeffs_full = np.load("landsat_fourier_coeffs.npy")

    ecostress_obs_time = float(ecos_df["time"][ecos_idx])  # seconds since epoch
    landsat_obs_time = float(landsat_df["time"][landsat_idx])  # days since epoch

    landsat_obs_time_seconds = landsat_obs_time * 86400  # days to seconds

    sin_annual_ecos = np.sin(
        2 * np.pi * ecostress_obs_time / 31536000
    )
    cos_annual_ecos = np.cos(2 * np.pi * ecostress_obs_time / 31536000)
    sin_diurnal_ecos = np.sin(
        2 * np.pi * ecostress_obs_time / 86400
    )
    cos_diurnal_ecos = np.cos(2 * np.pi * ecostress_obs_time / 86400)

    ecostress_mean = (
        ecostress_fourier_coeffs_full[:, :, 0]
        + ecostress_fourier_coeffs_full[:, :, 1] * sin_annual_ecos
        + ecostress_fourier_coeffs_full[:, :, 2] * cos_annual_ecos
        + ecostress_fourier_coeffs_full[:, :, 3] * sin_diurnal_ecos
        + ecostress_fourier_coeffs_full[:, :, 4] * cos_diurnal_ecos
    )

    sin_annual_landsat = np.sin(
        2 * np.pi * landsat_obs_time / 365.25
    )
    cos_annual_landsat = np.cos(2 * np.pi * landsat_obs_time / 365.25)

    landsat_mean = (
        landsat_fourier_coeffs_full[:, :, 0]
        + landsat_fourier_coeffs_full[:, :, 1] * sin_annual_landsat
        + landsat_fourier_coeffs_full[:, :, 2] * cos_annual_landsat
    )

    print(f"ECOSTRESS observation time: {ecostress_obs_time:.2f} seconds since epoch")
    print(
        f"Landsat observation time: {landsat_obs_time:.2f} days since epoch ({landsat_obs_time_seconds:.2f} seconds)"
    )
    print(f"ECOSTRESS mean image shape: {ecostress_mean.shape}")
    print(f"Landsat mean image shape: {landsat_mean.shape}")

    return ecostress_mean, landsat_mean


ecostress_mean, landsat_mean = compute_mean_images_at_observation_time(
    ecos_idx=500, landsat_idx=0
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

valid_ecos_mean = ecostress_mean[~np.isnan(ecostress_mean) & (ecostress_mean != 0)]
if len(valid_ecos_mean) > 0:
    im1 = axes[0].imshow(
        ecostress_mean,
        cmap="coolwarm",
        vmin=np.percentile(valid_ecos_mean, 5),
        vmax=np.percentile(valid_ecos_mean, 95),
    )
    plt.colorbar(im1, ax=axes[0], label="LST (K)")
axes[0].set_title("ECOSTRESS Mean at Observation Time")
axes[0].axis("off")

valid_landsat_mean = landsat_mean[~np.isnan(landsat_mean) & (landsat_mean != 0)]
if len(valid_landsat_mean) > 0:
    im2 = axes[1].imshow(
        landsat_mean,
        cmap="gray",
        vmin=np.percentile(valid_landsat_mean, 2),
        vmax=np.percentile(valid_landsat_mean, 98),
    )
    plt.colorbar(im2, ax=axes[1])
axes[1].set_title("Landsat Mean at Observation Time")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def reproject_ecostress_to_landsat(ecos_lst_2d):
    landsat_lst = np.zeros(landsat_shape, dtype=np.float32)

    reproject(
        source=ecos_lst_2d.filled(np.nan),
        destination=landsat_lst,
        src_transform=ecos_transform,
        src_crs=ecos_crs,
        dst_transform=landsat_transform,
        dst_crs=landsat_crs,
        resampling=Resampling.bilinear,
    )

    return landsat_lst


test_idx = 500
ecos_lst_original = ecos_df["LST"][test_idx, :, :]
landsat_lst_reprojected = reproject_ecostress_to_landsat(ecos_lst_original)

print(f"Reprojected ECOSTRESS shape: {landsat_lst_reprojected.shape}")
print(f"Should match Landsat: {landsat_shape}")

In [ ]:
def plot_alignment_check(ecos_idx=500, landsat_idx=0, dpi=300):
    ecos_lst_original = ecos_df["LST"][ecos_idx, :, :]
    ecos_data = ecos_lst_original.filled(np.nan)
    landsat_lst_reprojected = reproject_ecostress_to_landsat(ecos_lst_original)

    landsat_lst_reprojected[landsat_lst_reprojected == 0] = np.nan

    landsat_data = landsat_df["SR_B1"][landsat_idx, :, :].filled(np.nan)

    landsat_extent = [
        landsat_x[0] - 15,
        landsat_x[-1] + 15,
        landsat_y[-1] - 15,
        landsat_y[0] + 15,
    ]

    fig, axes = plt.subplots(1, 3, figsize=(30, 10), dpi=dpi)

    valid_ecos = landsat_lst_reprojected[~np.isnan(landsat_lst_reprojected)]
    im1 = axes[0].imshow(
        landsat_lst_reprojected,
        extent=landsat_extent,
        cmap="coolwarm",
        vmin=np.nanpercentile(valid_ecos, 5),
        vmax=np.nanpercentile(valid_ecos, 95),
        interpolation="nearest",
    )
    axes[0].set_title(f"ECOSTRESS LST (reprojected)\nIndex {ecos_idx}", fontsize=16)
    axes[0].set_xlabel("Easting (m)", fontsize=14)
    axes[0].set_ylabel("Northing (m)", fontsize=14)
    plt.colorbar(im1, ax=axes[0], label="LST (K)")

    valid_landsat = landsat_data[~np.isnan(landsat_data)]
    im2 = axes[1].imshow(
        landsat_data,
        extent=landsat_extent,
        cmap="gray",
        vmin=np.nanpercentile(valid_landsat, 2),
        vmax=np.nanpercentile(valid_landsat, 98),
        interpolation="nearest",
    )
    axes[1].set_title(f"Landsat SR_B1\nIndex {landsat_idx}", fontsize=16)
    axes[1].set_xlabel("Easting (m)", fontsize=14)
    axes[1].set_ylabel("Northing (m)", fontsize=14)
    plt.colorbar(im2, ax=axes[1], label="Surface Reflectance")

    axes[2].imshow(
        landsat_data,
        extent=landsat_extent,
        cmap="gray",
        alpha=0.7,
        vmin=np.nanpercentile(valid_landsat, 2),
        vmax=np.nanpercentile(valid_landsat, 98),
        interpolation="nearest",
    )
    axes[2].imshow(
        landsat_lst_reprojected,
        extent=landsat_extent,
        cmap="coolwarm",
        alpha=0.5,
        vmin=np.nanpercentile(valid_ecos, 5),
        vmax=np.nanpercentile(valid_ecos, 95),
        interpolation="nearest",
    )
    axes[2].set_title("Overlay (Landsat gray + ECOSTRESS color)", fontsize=16)
    axes[2].set_xlabel("Easting (m)", fontsize=14)
    axes[2].set_ylabel("Northing (m)", fontsize=14)

    plt.tight_layout()
    plt.savefig(
        f"alignment_check_ecos{ecos_idx}_landsat{landsat_idx}.png",
        dpi=dpi,
        bbox_inches="tight",
    )
    plt.show()

    print(f"Valid ECOSTRESS pixels: {np.sum(~np.isnan(landsat_lst_reprojected))}")
    print(
        f"ECOSTRESS value range: {np.nanmin(valid_ecos):.1f} - {np.nanmax(valid_ecos):.1f} K"
    )
    print(
        f"Saved high-res image to: alignment_check_ecos{ecos_idx}_landsat{landsat_idx}.png"
    )


plot_alignment_check(ecos_idx=500, landsat_idx=0, dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import rotate
from skimage import measure

def extract_and_align_nonmasked_region_with_params(masked_array):
    binary_mask = masked_array.mask
    valid_mask = ~binary_mask
    labeled_mask, num_features = measure.label(valid_mask, return_num=True)
    regions = measure.regionprops(labeled_mask)
    regions.sort(key=lambda x: x.area, reverse=True)
    largest_region = regions[0]

    angle_rad = largest_region.orientation
    angle_degrees = np.degrees(angle_rad)
    rotation_angle = -angle_degrees
    min_row, min_col, max_row, max_col = largest_region.bbox

    extracted_data = masked_array.data[min_row:max_row, min_col:max_col]
    extracted_mask = masked_array.mask[min_row:max_row, min_col:max_col]

    aligned_data = rotate(
        extracted_data,
        rotation_angle,
        resize=True,
        preserve_range=True,
        mode="constant",
        cval=0,
    )

    aligned_mask = (
        rotate(
            extracted_mask.astype(float),
            rotation_angle,
            resize=True,
            preserve_range=True,
            mode="constant",
            cval=1,
        )
        > 0.5
    )

    aligned_image = np.ma.array(aligned_data, mask=aligned_mask)
    aligned_image = aligned_image.filled(0).astype(masked_array.dtype)

    crop_bounds = None
    if np.any(aligned_mask):
        rows, cols = np.where(~aligned_mask)
        if len(rows) > 0 and len(cols) > 0:
            crop_min_row, crop_max_row = rows.min(), rows.max()
            crop_min_col, crop_max_col = cols.min(), cols.max()
            crop_bounds = (crop_min_row, crop_max_row, crop_min_col, crop_max_col)
            aligned_image = aligned_data[
                crop_min_row : crop_max_row + 1, crop_min_col : crop_max_col + 1
            ]

    params = {
        "bbox": (min_row, min_col, max_row, max_col),
        "rotation_angle": rotation_angle,
        "crop_bounds": crop_bounds,
    }

    print(f"Extracted region from [{min_row}:{max_row}, {min_col}:{max_col}]")
    print(f"Rotation angle applied: {rotation_angle:.2f} degrees")
    if crop_bounds is not None:
        print(
            f"Cropped to bounds: row[{crop_bounds[0]}:{crop_bounds[1]}], col[{crop_bounds[2]}:{crop_bounds[3]}]"
        )
        print(f"Final size: {aligned_image.shape}")

    return aligned_image, params


def apply_transformation(image_array, params):
    min_row, min_col, max_row, max_col = params["bbox"]
    rotation_angle = params["rotation_angle"]
    crop_bounds = params["crop_bounds"]

    extracted_data = image_array[min_row:max_row, min_col:max_col]

    aligned_data = rotate(
        extracted_data,
        rotation_angle,
        resize=True,
        preserve_range=True,
        mode="constant",
        cval=0,
    )

    if crop_bounds is not None:
        crop_min_row, crop_max_row, crop_min_col, crop_max_col = crop_bounds
        aligned_data = aligned_data[
            crop_min_row : crop_max_row + 1, crop_min_col : crop_max_col + 1
        ]

    return aligned_data


def plot_landsat_ecostress_comparison(ecos_idx=500, landsat_idx=0):
    ecos_lst_original = ecos_df["LST"][ecos_idx, :, :]
    landsat_lst_reprojected = reproject_ecostress_to_landsat(ecos_lst_original)
    landsat_lst_reprojected[landsat_lst_reprojected == 0] = np.nan

    landsat_data = landsat_df["SR_B5"][landsat_idx, :, :].filled(np.nan)

    landsat_masked = np.ma.array(landsat_data, mask=np.isnan(landsat_data))

    print("Extracting and aligning Landsat...")
    aligned_landsat, params = extract_and_align_nonmasked_region_with_params(
        landsat_masked
    )

    print("\nApplying same transformation to ECOSTRESS...")
    aligned_ecos = apply_transformation(
        np.nan_to_num(landsat_lst_reprojected, nan=0), params
    )

    landsat_sub = aligned_landsat[2400:2600, 2200:2400]
    ecos_sub = aligned_ecos[2400:2600, 2200:2400]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    valid_landsat = landsat_sub[landsat_sub != 0]
    im1 = axes[0].imshow(
        landsat_sub,
        cmap="gray",
        vmin=np.percentile(valid_landsat, 2),
        vmax=np.percentile(valid_landsat, 98),
    )
    axes[0].set_title(f"Landsat SR_B1\nIndex {landsat_idx}")
    axes[0].axis("off")
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

    valid_ecos = ecos_sub[ecos_sub != 0]
    if len(valid_ecos) > 0:
        im2 = axes[1].imshow(
            ecos_sub,
            cmap="coolwarm",
            vmin=np.percentile(valid_ecos, 5),
            vmax=np.percentile(valid_ecos, 95),
        )
        plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="LST (K)")
    else:
        axes[1].imshow(ecos_sub, cmap="coolwarm")
    axes[1].set_title(f"ECOSTRESS LST\nIndex {ecos_idx}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\nLandsat sub-region shape: {landsat_sub.shape}")
    print(f"ECOSTRESS sub-region shape: {ecos_sub.shape}")
    if len(valid_landsat) > 0:
        print(
            f"Landsat value range: {valid_landsat.min():.1f} - {valid_landsat.max():.1f}"
        )
    if len(valid_ecos) > 0:
        print(
            f"ECOSTRESS value range: {valid_ecos.min():.1f} - {valid_ecos.max():.1f} K"
        )

    return aligned_landsat, aligned_ecos, params


aligned_landsat, aligned_ecos, params = plot_landsat_ecostress_comparison(
    ecos_idx=500, landsat_idx=0
)

In [ ]:
def plot_original_mean_residual(ecos_idx=500, landsat_idx=0):
    print("Getting aligned images...")
    ecos_lst_original = ecos_df["LST"][ecos_idx, :, :]
    landsat_lst_reprojected = reproject_ecostress_to_landsat(ecos_lst_original)
    landsat_lst_reprojected[landsat_lst_reprojected == 0] = np.nan

    landsat_data = landsat_df["SR_B5"][landsat_idx, :, :].filled(np.nan)
    landsat_masked = np.ma.array(landsat_data, mask=np.isnan(landsat_data))

    print("Extracting and aligning...")
    aligned_landsat, params = extract_and_align_nonmasked_region_with_params(
        landsat_masked
    )
    aligned_ecos = apply_transformation(
        np.nan_to_num(landsat_lst_reprojected, nan=0), params
    )

    row_min, row_max = 2400, 2600
    col_min, col_max = 2200, 2400

    landsat_sub = aligned_landsat[row_min:row_max, col_min:col_max]
    ecos_sub = aligned_ecos[row_min:row_max, col_min:col_max]

    print("\nComputing Fourier means...")
    ecostress_mean_full, landsat_mean_full = compute_mean_images_at_observation_time(
        ecos_idx=ecos_idx, landsat_idx=landsat_idx
    )

    if hasattr(ecostress_mean_full, "filled"):
        ecostress_mean_reprojected = reproject_ecostress_to_landsat(ecostress_mean_full)
    else:
        ecostress_mean_masked = np.ma.array(
            ecostress_mean_full,
            mask=np.isnan(ecostress_mean_full) | (ecostress_mean_full == 0),
        )
        ecostress_mean_reprojected = reproject_ecostress_to_landsat(
            ecostress_mean_masked
        )

    ecostress_mean_reprojected[ecostress_mean_reprojected == 0] = np.nan

    print("Aligning mean images...")
    aligned_ecos_mean = apply_transformation(
        np.nan_to_num(ecostress_mean_reprojected, nan=0), params
    )
    aligned_landsat_mean = apply_transformation(
        np.nan_to_num(landsat_mean_full, nan=0), params
    )

    ecos_mean_sub = aligned_ecos_mean[row_min:row_max, col_min:col_max]
    landsat_mean_sub = aligned_landsat_mean[row_min:row_max, col_min:col_max]

    print("Computing residuals...")
    ecos_residual = ecos_sub.copy()
    ecos_residual[ecos_sub != 0] = (
        ecos_sub[ecos_sub != 0] - ecos_mean_sub[ecos_sub != 0]
    )

    landsat_residual = landsat_sub.copy()
    landsat_residual[landsat_sub != 0] = (
        landsat_sub[landsat_sub != 0] - landsat_mean_sub[landsat_sub != 0]
    )

    fig, axes = plt.subplots(3, 2, figsize=(14, 18))

    valid_landsat = landsat_sub[landsat_sub != 0]
    im1 = axes[0, 0].imshow(
        landsat_sub,
        cmap="gray",
        vmin=np.percentile(valid_landsat, 2),
        vmax=np.percentile(valid_landsat, 98),
    )
    axes[0, 0].set_title(
        f"Landsat Original\nIndex {landsat_idx}", fontsize=14, fontweight="bold"
    )
    axes[0, 0].axis("off")
    plt.colorbar(im1, ax=axes[0, 0], fraction=0.046, pad=0.04)

    valid_ecos = ecos_sub[ecos_sub != 0]
    ecos_vmin, ecos_vmax = (
        np.percentile(valid_ecos, [5, 95]) if len(valid_ecos) > 0 else (0, 1)
    )
    im2 = axes[0, 1].imshow(ecos_sub, cmap="coolwarm", vmin=ecos_vmin, vmax=ecos_vmax)
    axes[0, 1].set_title(
        f"ECOSTRESS Original\nIndex {ecos_idx}", fontsize=14, fontweight="bold"
    )
    axes[0, 1].axis("off")
    plt.colorbar(im2, ax=axes[0, 1], fraction=0.046, pad=0.04, label="LST (K)")

    valid_landsat_mean = landsat_mean_sub[landsat_mean_sub != 0]
    im3 = axes[1, 0].imshow(
        landsat_mean_sub,
        cmap="gray",
        vmin=np.percentile(valid_landsat_mean, 2) if len(valid_landsat_mean) > 0 else 0,
        vmax=(
            np.percentile(valid_landsat_mean, 98) if len(valid_landsat_mean) > 0 else 1
        ),
    )
    axes[1, 0].set_title(
        "Landsat Fourier Mean\n(at observation time)", fontsize=14, fontweight="bold"
    )
    axes[1, 0].axis("off")
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

    valid_ecos_mean = ecos_mean_sub[ecos_mean_sub != 0]
    im4 = axes[1, 1].imshow(
        ecos_mean_sub, cmap="coolwarm", vmin=ecos_vmin, vmax=ecos_vmax
    )
    axes[1, 1].set_title(
        "ECOSTRESS Fourier Mean\n(at observation time)", fontsize=14, fontweight="bold"
    )
    axes[1, 1].axis("off")
    plt.colorbar(im4, ax=axes[1, 1], fraction=0.046, pad=0.04, label="LST (K)")

    valid_landsat_res = landsat_residual[landsat_residual != 0]
    if len(valid_landsat_res) > 0:
        res_max = max(
            abs(np.percentile(valid_landsat_res, 2)),
            abs(np.percentile(valid_landsat_res, 98)),
        )
        im5 = axes[2, 0].imshow(
            landsat_residual, cmap="RdBu_r", vmin=-res_max, vmax=res_max
        )
        plt.colorbar(im5, ax=axes[2, 0], fraction=0.046, pad=0.04)
    axes[2, 0].set_title(
        "Landsat Residual\n(Original - Mean)", fontsize=14, fontweight="bold"
    )
    axes[2, 0].axis("off")

    valid_ecos_res = ecos_residual[ecos_residual != 0]
    if len(valid_ecos_res) > 0:
        ecos_res_max = max(
            abs(np.percentile(valid_ecos_res, 5)),
            abs(np.percentile(valid_ecos_res, 95)),
        )
        im6 = axes[2, 1].imshow(
            ecos_residual, cmap="RdBu_r", vmin=-ecos_res_max, vmax=ecos_res_max
        )
        plt.colorbar(im6, ax=axes[2, 1], fraction=0.046, pad=0.04, label="LST (K)")
    axes[2, 1].set_title(
        "ECOSTRESS Residual\n(Original - Mean)", fontsize=14, fontweight="bold"
    )
    axes[2, 1].axis("off")

    plt.tight_layout()
    plt.savefig(
        f"original_mean_residual_ecos{ecos_idx}_landsat{landsat_idx}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

    print(f"\n{'='*60}")
    print("STATISTICS")
    print(f"{'='*60}")
    print(f"\nLandsat (SR_B1):")
    print(f"  Original range: {valid_landsat.min():.2f} to {valid_landsat.max():.2f}")
    if len(valid_landsat_mean) > 0:
        print(
            f"  Mean range: {valid_landsat_mean.min():.2f} to {valid_landsat_mean.max():.2f}"
        )
    if len(valid_landsat_res) > 0:
        print(
            f"  Residual range: {valid_landsat_res.min():.2f} to {valid_landsat_res.max():.2f}"
        )
        print(f"  Residual std: {np.std(valid_landsat_res):.2f}")

    print(f"\nECOSTRESS (LST):")
    print(f"  Original range: {valid_ecos.min():.2f} to {valid_ecos.max():.2f} K")
    if len(valid_ecos_mean) > 0:
        print(
            f"  Mean range: {valid_ecos_mean.min():.2f} to {valid_ecos_mean.max():.2f} K"
        )
    if len(valid_ecos_res) > 0:
        print(
            f"  Residual range: {valid_ecos_res.min():.2f} to {valid_ecos_res.max():.2f} K"
        )
        print(f"  Residual std: {np.std(valid_ecos_res):.2f} K")

    print(f"\nSaved to: original_mean_residual_ecos{ecos_idx}_landsat{landsat_idx}.png")

    return (
        landsat_sub,
        ecos_sub,
        landsat_mean_sub,
        ecos_mean_sub,
        landsat_residual,
        ecos_residual,
    )


(landsat_orig, ecos_orig, landsat_mean, ecos_mean, landsat_res, ecos_res) = (
    plot_original_mean_residual(ecos_idx=500, landsat_idx=0)
)

In [ ]:
def plot_landsat_ecostress_comparison_with_crop(ecos_idx=500, landsat_idx=0, dpi=200):
    ecos_lst_original = ecos_df["LST"][ecos_idx, :, :]
    landsat_lst_reprojected = reproject_ecostress_to_landsat(ecos_lst_original)
    landsat_lst_reprojected[landsat_lst_reprojected == 0] = np.nan

    landsat_data = landsat_df["SR_B5"][landsat_idx, :, :].filled(np.nan)

    landsat_masked = np.ma.array(landsat_data, mask=np.isnan(landsat_data))

    print("Extracting and aligning Landsat...")
    aligned_landsat, params = extract_and_align_nonmasked_region_with_params(
        landsat_masked
    )

    print("\nApplying same transformation to ECOSTRESS...")
    aligned_ecos = apply_transformation(
        np.nan_to_num(landsat_lst_reprojected, nan=0), params
    )

    crop_row_start, crop_row_end = 2400, 2600
    crop_col_start, crop_col_end = 2200, 2400

    valid_landsat = aligned_landsat[aligned_landsat != 0]
    vmin_landsat = np.percentile(valid_landsat, 2)
    vmax_landsat = np.percentile(valid_landsat, 98)

    aligned_landsat_display = aligned_landsat.astype(float)
    aligned_landsat_display[aligned_landsat_display == 0] = np.nan

    fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=dpi)

    im1 = axes[0].imshow(
        aligned_landsat_display,
        cmap="gray",
        vmin=vmin_landsat,
        vmax=vmax_landsat,
        interpolation="nearest",
    )
    axes[0].set_title(f"Landsat SR_B1 (aligned)\nIndex {landsat_idx}", fontsize=14)
    axes[0].axis("off")

    from matplotlib.patches import Rectangle

    rect1 = Rectangle(
        (crop_col_start, crop_row_start),
        crop_col_end - crop_col_start,
        crop_row_end - crop_row_start,
        linewidth=3,
        edgecolor="red",
        facecolor="none",
    )
    axes[0].add_patch(rect1)
    axes[0].text(
        crop_col_start,
        crop_row_start - 20,
        "CROP REGION",
        color="red",
        fontsize=12,
        weight="bold",
    )
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

    valid_ecos = aligned_ecos[aligned_ecos != 0]
    vmin_ecos = np.percentile(valid_ecos, 5) if len(valid_ecos) > 0 else 0
    vmax_ecos = np.percentile(valid_ecos, 95) if len(valid_ecos) > 0 else 1

    aligned_ecos_display = aligned_ecos.astype(float)
    aligned_ecos_display[aligned_ecos_display == 0] = np.nan

    if len(valid_ecos) > 0:
        im2 = axes[1].imshow(
            aligned_ecos_display,
            cmap="coolwarm",
            vmin=vmin_ecos,
            vmax=vmax_ecos,
            interpolation="nearest",
        )
        plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="LST (K)")
    else:
        axes[1].imshow(aligned_ecos_display, cmap="coolwarm")
    axes[1].set_title(f"ECOSTRESS LST (aligned)\nIndex {ecos_idx}", fontsize=14)
    axes[1].axis("off")

    rect2 = Rectangle(
        (crop_col_start, crop_row_start),
        crop_col_end - crop_col_start,
        crop_row_end - crop_row_start,
        linewidth=3,
        edgecolor="red",
        facecolor="none",
    )
    axes[1].add_patch(rect2)
    axes[1].text(
        crop_col_start,
        crop_row_start - 20,
        "CROP REGION",
        color="red",
        fontsize=12,
        weight="bold",
    )

    plt.tight_layout()
    plt.savefig(
        f"aligned_with_crop_ecos{ecos_idx}_landsat{landsat_idx}.png",
        dpi=dpi,
        bbox_inches="tight",
    )
    plt.show()

    print(f"\nSaved to: aligned_with_crop_ecos{ecos_idx}_landsat{landsat_idx}.png")
    print(
        f"Full aligned image shapes - Landsat: {aligned_landsat.shape}, ECOSTRESS: {aligned_ecos.shape}"
    )
    print(
        f"Crop region: rows [{crop_row_start}:{crop_row_end}], cols [{crop_col_start}:{crop_col_end}]"
    )
    print(f"Landsat vmin/vmax: {vmin_landsat:.2f}/{vmax_landsat:.2f}")

    return aligned_landsat, aligned_ecos


aligned_landsat, aligned_ecos = plot_landsat_ecostress_comparison_with_crop(
    ecos_idx=500, landsat_idx=0, dpi=200
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio import features
from scipy.linalg import cho_factor, cho_solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter, binary_dilation
from scipy.optimize import minimize
from tqdm import tqdm
import pickle
import hashlib
from pathlib import Path


def get_cache_dir():
    cache_dir = Path("./gp_cache")
    cache_dir.mkdir(exist_ok=True)
    return cache_dir


def get_cache_key(*args, **kwargs):
    key_str = str(args) + str(sorted(kwargs.items()))
    return hashlib.md5(key_str.encode()).hexdigest()


def save_to_cache(data, cache_name, *args, **kwargs):
    cache_dir = get_cache_dir()
    cache_key = get_cache_key(*args, **kwargs)
    cache_file = cache_dir / f"{cache_name}_{cache_key}.pkl"

    with open(cache_file, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved to cache: {cache_file.name}")


def load_from_cache(cache_name, *args, **kwargs):
    cache_dir = get_cache_dir()
    cache_key = get_cache_key(*args, **kwargs)
    cache_file = cache_dir / f"{cache_name}_{cache_key}.pkl"

    if cache_file.exists():
        with open(cache_file, "rb") as f:
            data = pickle.load(f)
        print(f"Loaded from cache: {cache_file.name}")
        return data
    return None


def clear_cache():
    cache_dir = get_cache_dir()
    cache_files = list(cache_dir.glob("*.pkl"))
    for f in cache_files:
        f.unlink()
    print(f"Cleared {len(cache_files)} cache files")


def list_cache():
    cache_dir = get_cache_dir()
    cache_files = list(cache_dir.glob("*.pkl"))
    print(f"\nCache directory: {cache_dir}")
    print(f"Total files: {len(cache_files)}\n")

    total_size = 0
    for f in sorted(cache_files):
        size_mb = f.stat().st_size / (1024 * 1024)
        total_size += size_mb
        print(f"  {f.name}: {size_mb:.2f} MB")

    print(f"\nTotal cache size: {total_size:.2f} MB")


def squared_exponential_kernel(coords1, coords2, length_scale, signal_variance):
    sq_distances = cdist(coords1, coords2, metric="sqeuclidean")
    return signal_variance * np.exp(-0.5 * sq_distances / (length_scale**2))


def estimate_length_scale_from_landsat_multitemporal(
    landsat_region_data_list, coords_list, noise_variance=5e-4
):
    T = len(landsat_region_data_list)

    total_points = sum(len(y.ravel()) for y in landsat_region_data_list)
    if total_points < 5:
        return 2.0

    y_list = []
    coords_processed = []
    for t in range(T):
        y_t = landsat_region_data_list[t].ravel()
        coords_t = coords_list[t]
        n_t = len(y_t)

        if n_t == 0:
            continue

        if n_t > 200:
            idx = np.random.choice(n_t, 200, replace=False)
            y_t = y_t[idx]
            coords_t = coords_t[idx]

        y_list.append(y_t)
        coords_processed.append(coords_t)

    if len(y_list) == 0:
        return 2.0

    sq_dists_list = [cdist(c, c, metric="sqeuclidean") for c in coords_processed]
    n_list = [len(y) for y in y_list]

    def neg_joint_log_likelihood(params):
        ls = np.exp(params[0])
        signal_var = np.exp(params[1])

        total_nll = 0.0

        for t in range(len(y_list)):
            y_t = y_list[t]
            sq_dists_t = sq_dists_list[t]
            n_t = n_list[t]

            K_t = signal_var * np.exp(-0.5 * sq_dists_t / (ls**2))
            K_t[np.diag_indices(n_t)] += noise_variance

            try:
                L_t, _ = cho_factor(K_t)
                alpha_t = cho_solve((L_t, False), y_t)
                log_det_t = 2.0 * np.sum(np.log(np.diag(L_t)))
                nll_t = 0.5 * (log_det_t + y_t @ alpha_t)
                total_nll += nll_t
            except:
                return 1e10

        return total_nll

    all_sq_dists = np.concatenate([sd[sd > 0] for sd in sq_dists_list])
    if len(all_sq_dists) > 0:
        init_ls = np.sqrt(np.median(all_sq_dists)) / 2
    else:
        init_ls = 5.0

    all_values = np.concatenate(y_list)
    init_var = np.var(all_values) if np.var(all_values) > 0 else 1.0

    result = minimize(
        neg_joint_log_likelihood,
        x0=[np.log(init_ls), np.log(init_var)],
        bounds=[
            (np.log(0.5), np.log(50.0)),
            (np.log(1e-6), np.log(1e6)),
        ],
        method="L-BFGS-B",
    )

    if result.success:
        ls = np.exp(result.x[0])
        return ls
    else:
        return init_ls


def load_mask_from_geojson(geojson_path, image_shape, transform, use_cache=True):
    if use_cache:
        cached = load_from_cache("mask", geojson_path, image_shape, str(transform))
        if cached is not None:
            return cached

    gdf = gpd.read_file(geojson_path)

    mask = np.zeros(image_shape, dtype=np.int32)
    for idx, geom in enumerate(gdf.geometry):
        if geom and not geom.is_empty:
            rasterized = features.rasterize(
                [(geom, idx + 1)],
                out_shape=image_shape,
                transform=transform,
                dtype=np.int32,
            )
            mask = np.maximum(mask, rasterized)

    if use_cache:
        save_to_cache(mask, "mask", geojson_path, image_shape, str(transform))

    return mask


def estimate_landsat_length_scales_multitemporal(
    landsat_residual_list, region_mask, noise_variance=5e-4, use_cache=True
):
    data_hashes = [
        hashlib.md5(lr.tobytes()).hexdigest()[:8] for lr in landsat_residual_list
    ]
    mask_hash = hashlib.md5(region_mask.tobytes()).hexdigest()[:8]
    if use_cache:
        cached = load_from_cache(
            "landsat_length_scales_multitemporal",
            str(data_hashes),
            mask_hash,
            noise_variance,
        )
        if cached is not None:
            return cached

    length_scales = {}
    regions = np.unique(region_mask)
    regions = regions[regions != 0]

    T = len(landsat_residual_list)
    print(
        f"\nEstimating length scales from {T} Landsat images for {len(regions)} regions..."
    )
    print(
        f"  (Joint likelihood with temporal independence, noise_var={noise_variance})"
    )

    for region_id in tqdm(regions):
        region_bool_mask = region_mask == region_id

        values_list = []
        coords_list = []

        for t in range(T):
            values_t = landsat_residual_list[t][region_bool_mask]
            valid_values_t = values_t[~np.isnan(values_t) & (values_t != 0)]

            if len(valid_values_t) >= 5:
                coords_t = np.column_stack(np.where(region_bool_mask))
                valid_mask_t = ~np.isnan(values_t) & (values_t != 0)
                coords_t = coords_t[valid_mask_t]

                values_list.append(valid_values_t)
                coords_list.append(coords_t)

        if len(values_list) == 0:
            length_scales[region_id] = 2.0
            continue

        ls = estimate_length_scale_from_landsat_multitemporal(
            values_list, coords_list, noise_variance
        )
        length_scales[region_id] = ls

    if use_cache:
        save_to_cache(
            length_scales,
            "landsat_length_scales_multitemporal",
            str(data_hashes),
            mask_hash,
            noise_variance,
        )

    return length_scales


def estimate_ecostress_region_parameters_multitemporal(
    ecostress_residual_list, region_mask, length_scales, blur_sigma, noise_var, use_cache=True
):
    ecos_hashes = [
        hashlib.md5(er.tobytes()).hexdigest()[:8] for er in ecostress_residual_list
    ]
    mask_hash = hashlib.md5(region_mask.tobytes()).hexdigest()[:8]
    ls_hash = hashlib.md5(str(sorted(length_scales.items())).encode()).hexdigest()[:8]

    if use_cache:
        cached = load_from_cache(
            "ecostress_parameters_multitemporal", str(ecos_hashes), mask_hash, ls_hash
        )
        if cached is not None:
            return cached

    parameters = {}
    T = len(ecostress_residual_list)

    print(f"\nEstimating ECOSTRESS variances from {T} images...")
    print(f"  (Computing variance per image, then taking median)")

    for region_id in np.unique(region_mask):
        if region_id == 0:
            continue

        region_bool_mask = region_mask == region_id

        variances = []
        for t in range(T):
            values_t = ecostress_residual_list[t][region_bool_mask]
            valid_values_t = values_t[~np.isnan(values_t) & (values_t != 0)]

            if len(valid_values_t) > 0:
                length_scale = length_scales.get(region_id, 2.0)
                var_t = (np.var(valid_values_t) - noise_var) * (length_scale**2 + blur_sigma**2) / (length_scale**2)
                variances.append(var_t)

        if len(variances) == 0:
            parameters[region_id] = (1e-6, 2.0)
            continue

        median_variance = np.median(variances)
        ls = length_scales.get(region_id, 2.0)
        signal_var = max(median_variance, 1e-6)

        parameters[region_id] = (signal_var, ls)

    if use_cache:
        save_to_cache(
            parameters,
            "ecostress_parameters_multitemporal",
            str(ecos_hashes),
            mask_hash,
            ls_hash,
        )

    return parameters


def build_blur_weights(blur_sigma, radius=None):
    if radius is None:
        radius = int(np.ceil(3 * blur_sigma))

    x = np.arange(-radius, radius + 1)
    y = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(x, y)

    weights = np.exp(-(xx**2 + yy**2) / (2 * blur_sigma**2))
    weights /= weights.sum()

    offsets = []
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            if weights[i, j] > 1e-6:
                offsets.append((yy[i, j], xx[i, j], weights[i, j]))

    return offsets


def build_blur_matrix_row(
    obs_idx, obs_coords, latent_coords, blur_offsets, image_shape
):
    obs_i, obs_j = obs_coords[obs_idx]
    n_latent = len(latent_coords)
    row = np.zeros(n_latent)

    coord_to_idx = {(int(c[0]), int(c[1])): idx for idx, c in enumerate(latent_coords)}

    for di, dj, weight in blur_offsets:
        src_i = obs_i + di
        src_j = obs_j + dj

        if 0 <= src_i < image_shape[0] and 0 <= src_j < image_shape[1]:
            if (src_i, src_j) in coord_to_idx:
                latent_idx = coord_to_idx[(src_i, src_j)]
                row[latent_idx] = weight

    return row


def get_neighborhood_pixels(target_mask, region_mask, dilation_radius):
    obs_mask = binary_dilation(target_mask, iterations=dilation_radius)
    obs_coords = np.column_stack(np.where(obs_mask))
    obs_regions = region_mask[obs_mask]

    return obs_coords, obs_regions, obs_mask


def build_block_diagonal_covariance(coords, regions, region_parameters):
    n = len(coords)
    K = np.zeros((n, n))

    for region_id in np.unique(regions):
        if region_id not in region_parameters:
            continue

        var, ls = region_parameters[region_id]

        region_mask = regions == region_id
        region_indices = np.where(region_mask)[0]

        if len(region_indices) == 0:
            continue

        region_coords = coords[region_indices]
        K_region = squared_exponential_kernel(region_coords, region_coords, ls, var)

        K[np.ix_(region_indices, region_indices)] = K_region

    return K


def conditional_simulation_region(
    region_id,
    ecostress_blurred_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
    n_samples=1,
):
    target_mask = region_mask == region_id
    if not target_mask.any():
        return None, None

    target_coords = np.column_stack(np.where(target_mask))
    image_shape = region_mask.shape
    n_target = len(target_coords)

    dilation_radius = int(np.ceil(4 * blur_sigma))
    latent_coords, latent_regions, obs_mask = get_neighborhood_pixels(
        target_mask, region_mask, dilation_radius
    )

    n_latent = len(latent_coords)

    obs_values = ecostress_blurred_residual[obs_mask]
    valid_obs = ~np.isnan(obs_values) & (obs_values != 0)

    obs_coords_valid = latent_coords[valid_obs]
    obs_values_valid = obs_values[valid_obs]
    obs_regions_valid = latent_regions[valid_obs]
    n_obs = len(obs_coords_valid)

    if n_obs == 0:
        return None, None

    K_latent = build_block_diagonal_covariance(
        latent_coords, latent_regions, region_parameters
    )

    blur_offsets = build_blur_weights(blur_sigma)

    W_obs = np.zeros((n_obs, n_latent))
    for obs_idx in range(n_obs):
        obs_coord = obs_coords_valid[obs_idx]
        latent_idx = np.where((latent_coords == obs_coord).all(axis=1))[0][0]
        W_obs[obs_idx, :] = build_blur_matrix_row(
            latent_idx, latent_coords, latent_coords, blur_offsets, image_shape
        )

    K_obs = W_obs @ K_latent @ W_obs.T
    K_obs[np.diag_indices(n_obs)] += noise_variance

    target_indices = []
    for tc in target_coords:
        matches = np.where((latent_coords == tc).all(axis=1))[0]
        if len(matches) > 0:
            target_indices.append(matches[0])
        else:
            target_indices.append(-1)

    target_indices = np.array(target_indices)
    valid_target = target_indices >= 0

    if not valid_target.any():
        return None, None

    K_cross = np.zeros((n_target, n_obs))
    for i, tidx in enumerate(target_indices):
        if tidx >= 0:
            K_cross[i, :] = K_latent[tidx, :] @ W_obs.T

    target_var, _ = region_parameters[region_id]

    try:
        L, _ = cho_factor(K_obs)
        alpha = cho_solve((L, False), obs_values_valid)

        posterior_mean = np.full(n_target, np.nan)
        posterior_mean[valid_target] = K_cross[valid_target] @ alpha

        samples = np.zeros((n_samples, n_target))

        if n_samples > 0:
            valid_target_indices = target_indices[valid_target]
            K_target = K_latent[np.ix_(valid_target_indices, valid_target_indices)]

            v_full = cho_solve((L, False), K_cross[valid_target].T)
            posterior_cov = K_target - K_cross[valid_target] @ v_full

            posterior_cov = 0.5 * (posterior_cov + posterior_cov.T)
            stabilization = max(1e-6, 1e-5 * target_var)
            posterior_cov[np.diag_indices(valid_target.sum())] += stabilization

            try:
                L_post = np.linalg.cholesky(posterior_cov)
                for i in range(n_samples):
                    z = np.random.randn(valid_target.sum())
                    samples[i] = np.full(n_target, np.nan)
                    samples[i, valid_target] = posterior_mean[valid_target] + L_post @ z
            except np.linalg.LinAlgError:
                print(
                    f"Warning: Cholesky failed for region {region_id}, using mean as samples"
                )
                for i in range(n_samples):
                    samples[i] = posterior_mean.copy()

        return samples, target_coords

    except np.linalg.LinAlgError:
        print(f"LinAlgError in region {region_id}")
        return None, None


def simulate_conditional(
    ecostress_blurred_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
    n_samples=5,
    max_region_size=5000,
):
    H, W = region_mask.shape
    simulations = np.zeros((n_samples, H, W))

    regions = np.unique(region_mask)
    regions = regions[regions != 0]

    print(
        f"Generating {n_samples} conditional simulations for {len(regions)} regions..."
    )
    for region_id in tqdm(regions):
        n_pixels = np.sum(region_mask == region_id)
        if n_pixels == 0 or n_pixels > max_region_size:
            continue

        samples, coords = conditional_simulation_region(
            region_id,
            ecostress_blurred_residual,
            region_mask,
            region_parameters,
            blur_sigma,
            noise_variance,
            n_samples,
        )

        if samples is not None and coords is not None:
            for i in range(n_samples):
                simulations[i, coords[:, 0], coords[:, 1]] = samples[i]

    return simulations


def plot_simulations(simulations, ecos_blurred_residual, title_prefix="Simulation"):
    n_samples = simulations.shape[0]
    fig, axes = plt.subplots(1, n_samples + 1, figsize=(5 * (n_samples + 1), 5))

    valid = ecos_blurred_residual[
        ~np.isnan(ecos_blurred_residual) & (ecos_blurred_residual != 0)
    ]
    if len(valid) > 0:
        vmin, vmax = np.percentile(valid, [5, 95])
        im = axes[0].imshow(
            ecos_blurred_residual, cmap="coolwarm", vmin=vmin, vmax=vmax
        )
        plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    axes[0].set_title("ECOSTRESS Residual Observed", fontsize=12)
    axes[0].axis("off")

    for i in range(n_samples):
        sim = simulations[i]
        valid_sim = sim[sim != 0]
        if len(valid_sim) > 0:
            im = axes[i + 1].imshow(sim, cmap="coolwarm", vmin=vmin, vmax=vmax)
            plt.colorbar(im, ax=axes[i + 1], fraction=0.046, pad=0.04)
        axes[i + 1].set_title(f"{title_prefix} {i+1}", fontsize=12)
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()


def plot_parameter_maps(region_mask, region_parameters, title="GP Parameters"):
    variance_map = np.full(region_mask.shape, np.nan)
    length_scale_map = np.full(region_mask.shape, np.nan)
    std_map = np.full(region_mask.shape, np.nan)

    for region_id, (var, ls) in region_parameters.items():
        mask = region_mask == region_id
        variance_map[mask] = var
        length_scale_map[mask] = ls
        std_map[mask] = np.sqrt(var)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    im1 = axes[0].imshow(variance_map, cmap="viridis")
    axes[0].set_title(f"{title}: Signal Variance", fontsize=14, fontweight="bold")
    axes[0].axis("off")
    cbar1 = plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    cbar1.set_label("Variance (K²)", fontsize=11)

    im2 = axes[1].imshow(std_map, cmap="viridis")
    axes[1].set_title(f"{title}: Signal Std Dev", fontsize=14, fontweight="bold")
    axes[1].axis("off")
    cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    cbar2.set_label("Std Dev (K)", fontsize=11)

    im3 = axes[2].imshow(length_scale_map, cmap="plasma")
    axes[2].set_title(f"{title}: Length Scale", fontsize=14, fontweight="bold")
    axes[2].axis("off")
    cbar3 = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    cbar3.set_label("Length Scale (pixels)", fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*60}")
    print(f"{title.upper()} - PARAMETER STATISTICS")
    print(f"{'='*60}")
    print(
        f"Variance range: [{np.nanmin(variance_map):.4f}, {np.nanmax(variance_map):.4f}]"
    )
    print(f"Std Dev range:  [{np.nanmin(std_map):.4f}, {np.nanmax(std_map):.4f}]")
    print(
        f"Length Scale range: [{np.nanmin(length_scale_map):.2f}, {np.nanmax(length_scale_map):.2f}]"
    )


def process_multitemporal_with_simulation(
    aligned_landsat_full_list,
    aligned_ecos_full_list,
    blur_sigma=3.57,
    noise_variance=1.0,
    landsat_noise_variance=5e-4,
    n_samples=5,
    max_region_size=5000,
    use_cache=True,
):
    landsat_train = aligned_landsat_full_list[:-1]
    ecos_train = aligned_ecos_full_list[:-1]

    landsat_test = aligned_landsat_full_list[-1]
    ecos_test = aligned_ecos_full_list[-1]

    print(f"Using {len(landsat_train)} observations for parameter estimation")
    print(f"Using 1 observation for conditional simulation")

    print("\nLoading segmentation mask...")
    with rasterio.open("input.tif") as src:
        transform = src.transform
        full_shape = landsat_test.shape

    mask_sub = load_mask_from_geojson(
        "masks_cleaned.geojson", full_shape, transform, use_cache=use_cache
    )

    print(f"Image shape: {landsat_test.shape}")
    print(f"Mask has {len(np.unique(mask_sub))} unique regions")

    length_scales = estimate_landsat_length_scales_multitemporal(
        landsat_train,
        mask_sub,
        noise_variance=landsat_noise_variance,
        use_cache=use_cache,
    )

    region_parameters = estimate_ecostress_region_parameters_multitemporal(
        ecos_train, mask_sub, length_scales, blur_sigma, noise_variance, use_cache=use_cache
    )

    plot_parameter_maps(mask_sub, region_parameters, title="ECOSTRESS GP")

    print(f"\n{'='*80}")
    print("LEARNED PARAMETERS FOR EACH REGION")
    print(f"{'='*80}")
    print(f"{'Region':<8} {'Pixels':<10} {'Length Scale':<15} {'Signal Variance':<18}")
    print(f"{'ID':<8} {'Count':<10} {'(pixels)':<15} {'(K²)':<18}")
    print("-" * 80)
    for rid, (var, ls) in sorted(region_parameters.items()):
        n_pixels = np.sum(mask_sub == rid)
        print(f"{rid:<8} {n_pixels:<10} {ls:<15.2f} {var:<18.4f}")

    simulations = simulate_conditional(
        ecos_test,
        mask_sub,
        region_parameters,
        blur_sigma,
        noise_variance,
        n_samples,
        max_region_size,
    )

    plot_simulations(simulations, ecos_test, title_prefix="Conditional Sample")

    return simulations, ecos_test, mask_sub, region_parameters


ecos_indices = [82, 300, 350, 500]
landsat_indices = [0, 114, 171, 210]

landsat_res_list = []
ecos_res_list = []

print("Loading temporal observations...")
print(f"ECOSTRESS indices: {ecos_indices}")
print(f"Landsat indices: {landsat_indices}")

for ecos_idx, landsat_idx in zip(ecos_indices, landsat_indices):
    print(f"\nLoading pair: ECOSTRESS[{ecos_idx}], Landsat[{landsat_idx}]")
    (landsat_orig, ecos_orig, landsat_mean, ecos_mean, landsat_res, ecos_res) = (
        plot_original_mean_residual(ecos_idx=ecos_idx, landsat_idx=landsat_idx)
    )
    landsat_res_list.append(landsat_res)
    ecos_res_list.append(ecos_res)

print(f"\nLoaded {len(landsat_res_list)} temporal observations")

simulations, ecos_sub, mask_sub, region_parameters = (
    process_multitemporal_with_simulation(
        aligned_landsat_full_list=landsat_res_list,
        aligned_ecos_full_list=ecos_res_list,
        blur_sigma=0.97 * 7/3,
        noise_variance=0.01,
        n_samples=5,
        max_region_size=5000,
        use_cache=True,
    )
)

In [ ]:
def reconstruct_region(
    region_id,
    ecostress_blurred_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
):
    target_mask = region_mask == region_id
    if not target_mask.any():
        return None, None, None

    target_coords = np.column_stack(np.where(target_mask))
    image_shape = region_mask.shape
    n_target = len(target_coords)

    dilation_radius = int(np.ceil(4 * blur_sigma))
    latent_coords, latent_regions, obs_mask = get_neighborhood_pixels(
        target_mask, region_mask, dilation_radius
    )

    n_latent = len(latent_coords)

    obs_values = ecostress_blurred_residual[obs_mask]
    valid_obs = ~np.isnan(obs_values) & (obs_values != 0)

    obs_coords_valid = latent_coords[valid_obs]
    obs_values_valid = obs_values[valid_obs]
    obs_regions_valid = latent_regions[valid_obs]
    n_obs = len(obs_coords_valid)

    if n_obs == 0:
        return None, None, None

    K_latent = build_block_diagonal_covariance(
        latent_coords, latent_regions, region_parameters
    )

    blur_offsets = build_blur_weights(blur_sigma)

    W_obs = np.zeros((n_obs, n_latent))
    for obs_idx in range(n_obs):
        obs_coord = obs_coords_valid[obs_idx]
        latent_idx = np.where((latent_coords == obs_coord).all(axis=1))[0][0]
        W_obs[obs_idx, :] = build_blur_matrix_row(
            latent_idx, latent_coords, latent_coords, blur_offsets, image_shape
        )

    K_obs = W_obs @ K_latent @ W_obs.T
    K_obs[np.diag_indices(n_obs)] += noise_variance

    target_indices = []
    for tc in target_coords:
        matches = np.where((latent_coords == tc).all(axis=1))[0]
        if len(matches) > 0:
            target_indices.append(matches[0])
        else:
            target_indices.append(-1)

    target_indices = np.array(target_indices)
    valid_target = target_indices >= 0

    if not valid_target.any():
        return None, None, None

    K_cross = np.zeros((n_target, n_obs))
    for i, tidx in enumerate(target_indices):
        if tidx >= 0:
            K_cross[i, :] = K_latent[tidx, :] @ W_obs.T

    target_var, _ = region_parameters[region_id]

    try:
        L, _ = cho_factor(K_obs)
        alpha = cho_solve((L, False), obs_values_valid)

        posterior_mean = np.full(n_target, np.nan)
        posterior_mean[valid_target] = K_cross[valid_target] @ alpha

        v = cho_solve((L, False), K_cross[valid_target].T)
        posterior_var = target_var - np.sum(K_cross[valid_target] * v.T, axis=1)
        posterior_var = np.maximum(posterior_var, 1e-10)

        posterior_std = np.full(n_target, np.nan)
        posterior_std[valid_target] = np.sqrt(posterior_var)

        return posterior_mean, posterior_std, target_coords

    except np.linalg.LinAlgError:
        print(f"LinAlgError in region {region_id}")
        return None, None, None


def compute_reconstruction(
    ecostress_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
    max_region_size=5000,
):
    H, W = region_mask.shape
    posterior_mean = np.zeros((H, W))
    posterior_std = np.full((H, W), np.nan)

    regions = np.unique(region_mask)
    regions = regions[regions != 0]

    print(f"\nComputing reconstruction for {len(regions)} regions...")
    for region_id in tqdm(regions):
        n_pixels = np.sum(region_mask == region_id)
        if n_pixels == 0 or n_pixels > max_region_size:
            continue

        mean, std, coords = reconstruct_region(
            region_id,
            ecostress_residual,
            region_mask,
            region_parameters,
            blur_sigma,
            noise_variance,
        )

        if mean is not None and coords is not None:
            posterior_mean[coords[:, 0], coords[:, 1]] = mean
            posterior_std[coords[:, 0], coords[:, 1]] = std

    return posterior_mean, posterior_std


posterior_mean, posterior_std = compute_reconstruction(
    ecos_sub,
    mask_sub,
    region_parameters,
    blur_sigma=0.97 * 7/3,
    noise_variance=0.1,
    max_region_size=5000,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

valid_obs = ecos_sub[~np.isnan(ecos_sub) & (ecos_sub != 0)]
if len(valid_obs) > 0:
    vmin, vmax = np.percentile(valid_obs, [5, 95])
    im1 = axes[0].imshow(ecos_sub, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="Residual (K)")
axes[0].set_title("ECOSTRESS Residual (Blurred)", fontsize=14, fontweight="bold")
axes[0].axis("off")

valid_mean = posterior_mean[~np.isnan(posterior_mean) & (posterior_mean != 0)]
if len(valid_mean) > 0:
    im2 = axes[1].imshow(posterior_mean, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="Residual (K)")
axes[1].set_title("Posterior Mean (Conditional Mean)", fontsize=14, fontweight="bold")
axes[1].axis("off")

valid_std = posterior_std[~np.isnan(posterior_std)]
if len(valid_std) > 0:
    im3 = axes[2].imshow(posterior_std * 2, cmap="magma", vmin=0)
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="Std Dev (K)")
axes[2].set_title(
    "Kriging Uncertainty", fontsize=14, fontweight="bold"
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("RECONSTRUCTION STATISTICS")
print(f"{'='*60}")
valid_mean = posterior_mean[~np.isnan(posterior_mean) & (posterior_mean != 0)]
valid_std = posterior_std[~np.isnan(posterior_std)]
if len(valid_mean) > 0:
    print(f"Kriging Mean range: [{np.min(valid_mean):.4f}, {np.max(valid_mean):.4f}]")
    print(f"Kriging Mean std: {np.std(valid_mean):.4f}")
if len(valid_std) > 0:
    print(
        f"Kriging Std Dev range: [{np.min(valid_std):.4f}, {np.max(valid_std):.4f}]"
    )
    print(f"Kriging Std Dev mean: {np.mean(valid_std):.4f}")

In [ ]:
valid_obs = ecos_sub[~np.isnan(ecos_sub) & (ecos_sub != 0)]
if len(valid_obs) > 0:
    vmax = max(abs(np.percentile(valid_obs, 5)), abs(np.percentile(valid_obs, 95)))
    vmin = -vmax
    im1 = plt.imshow(ecos_sub, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="Residual (K)")
plt.colorbar(label="Residual (°K)")
plt.axis("off")
plt.savefig("ecostress-residual.png", dpi=200)

plt.figure()

ecostress_mean_full, _ = compute_mean_images_at_observation_time(
    ecos_idx=500, landsat_idx=0
)

if hasattr(ecostress_mean_full, "filled"):
    ecostress_mean_reprojected = reproject_ecostress_to_landsat(ecostress_mean_full)
else:
    ecostress_mean_masked = np.ma.array(
        ecostress_mean_full,
        mask=np.isnan(ecostress_mean_full) | (ecostress_mean_full == 0),
    )
    ecostress_mean_reprojected = reproject_ecostress_to_landsat(ecostress_mean_masked)

ecostress_mean_reprojected[ecostress_mean_reprojected == 0] = np.nan

aligned_ecos_mean = apply_transformation(
    np.nan_to_num(ecostress_mean_reprojected, nan=0), params
)

row_min, row_max = 2400, 2600
col_min, col_max = 2200, 2400
ecos_mean_sub = aligned_ecos_mean[row_min:row_max, col_min:col_max]

original_ecos = ecos_sub + ecos_mean_sub

original_ecos_masked = original_ecos.copy()
original_ecos_masked[original_ecos == 0] = np.nan

plt.figure(figsize=(6, 5))
valid_orig = original_ecos_masked[~np.isnan(original_ecos_masked)]
if len(valid_orig) > 0:
    im = plt.imshow(
        original_ecos_masked,
        cmap="viridis",
        vmin=np.percentile(valid_orig, 5),
        vmax=np.percentile(valid_orig, 95),
    )
    plt.colorbar(im, fraction=0.046, pad=0.04, label="LST (K)")
plt.axis("off")
plt.savefig(
    "original-ecostress-t500.png", dpi=200, bbox_inches="tight", facecolor="white"
)
plt.show()

print(
    f"\nOriginal ECOSTRESS range: [{np.min(valid_orig):.2f}, {np.max(valid_orig):.2f}] K"
)

In [ ]:
landsat_idx = 1
landsat_residual = landsat_res_list[landsat_idx]

valid_landsat_res = landsat_residual[
    ~np.isnan(landsat_residual) & (landsat_residual != 0)
]
if len(valid_landsat_res) > 0:
    vmax = max(
        abs(np.percentile(valid_landsat_res, 5)),
        abs(np.percentile(valid_landsat_res, 95)),
    )
    vmin = -vmax

    plt.figure(figsize=(6, 5))
    im1 = plt.imshow(landsat_residual, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im1, fraction=0.046, pad=0.04, label="Residual")
    plt.axis("off")
    plt.savefig(
        f"landsat-residual-t{landsat_indices[landsat_idx]}.png",
        dpi=200,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()

_, landsat_mean_full = compute_mean_images_at_observation_time(
    ecos_idx=500, landsat_idx=landsat_indices[landsat_idx]
)

aligned_landsat_mean = apply_transformation(
    np.nan_to_num(landsat_mean_full, nan=0), params
)

row_min, row_max = 2400, 2600
col_min, col_max = 2200, 2400
landsat_mean_sub = aligned_landsat_mean[row_min:row_max, col_min:col_max]

original_landsat = landsat_residual + landsat_mean_sub

original_landsat_masked = original_landsat.copy()
original_landsat_masked[original_landsat == 0] = np.nan

plt.figure(figsize=(6, 5))
valid_orig_landsat = original_landsat_masked[~np.isnan(original_landsat_masked)]
if len(valid_orig_landsat) > 0:
    im = plt.imshow(
        original_landsat_masked,
        cmap="viridis",
        vmin=np.percentile(valid_orig_landsat, 2),
        vmax=np.percentile(valid_orig_landsat, 98),
    )
    plt.colorbar(im, fraction=0.046, pad=0.04, label="SR_B5")
plt.axis("off")
plt.savefig(
    f"original-landsat-t{landsat_indices[landsat_idx]}.png",
    dpi=200,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

print(
    f"\nOriginal Landsat range: [{np.min(valid_orig_landsat):.2f}, {np.max(valid_orig_landsat):.2f}]"
)

In [ ]:
valid_obs = ecos_sub[~np.isnan(ecos_sub) & (ecos_sub != 0)]
if len(valid_obs) > 0:
    vmax = max(abs(np.percentile(valid_obs, 5)), abs(np.percentile(valid_obs, 95)))
    vmin = -vmax

    posterior_mean_masked = posterior_mean.copy()
    posterior_mean_masked[posterior_mean_masked == 0] = np.nan

    im1 = plt.imshow(posterior_mean_masked, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im1, fraction=0.046, pad=0.04, label="Prediction (°K)")
plt.axis("off")
plt.savefig("posterior-mean.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

valid_std = posterior_std[~np.isnan(posterior_std)]
if len(valid_std) > 0:
    uncertainty_masked = posterior_std.copy() * 2
    uncertainty_masked[np.isnan(posterior_std)] = np.nan

    im2 = plt.imshow(uncertainty_masked, cmap="magma", vmin=0, vmax=0.8)
    plt.colorbar(im2, fraction=0.046, pad=0.04, label="Uncertainty (°K)")
plt.axis("off")
plt.savefig("uncertainty.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
ecos_var_map = np.full(mask_sub.shape, np.nan)
for region_id, (ecos_var, ls) in region_parameters.items():
    if region_id == 0:
        continue
    mask = mask_sub == region_id
    ecos_var_map[mask] = ecos_var

ecos_var_map[ecos_var_map == 0] = np.nan

im1 = plt.imshow(ecos_var_map, cmap="viridis")
plt.colorbar(im1, fraction=0.046, pad=0.04, label="Variance (°K²)")
plt.axis("off")
plt.savefig("ecostress-variance.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

landsat_var_map = np.full(mask_sub.shape, np.nan)
for region_id in np.unique(mask_sub):
    if region_id == 0:
        continue
    region_bool_mask = mask_sub == region_id

    region_values_temporal = []
    for landsat_img in landsat_res_list:
        values = landsat_img[region_bool_mask]
        valid_values = values[~np.isnan(values) & (values != 0)]
        region_values_temporal.extend(valid_values)

    if len(region_values_temporal) > 0:
        landsat_var_map[region_bool_mask] = np.var(region_values_temporal)

landsat_var_map[landsat_var_map == 0] = np.nan

im2 = plt.imshow(landsat_var_map, cmap="viridis")
plt.colorbar(im2, fraction=0.046, pad=0.04, label="Variance")
plt.axis("off")
plt.savefig("landsat-variance.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

length_scale_map = np.full(mask_sub.shape, np.nan)
for region_id, (ecos_var, ls) in region_parameters.items():
    if region_id == 0:
        continue
    mask = mask_sub == region_id
    length_scale_map[mask] = ls

length_scale_map[length_scale_map == 0] = np.nan

im3 = plt.imshow(length_scale_map, cmap="plasma")
plt.colorbar(im3, fraction=0.046, pad=0.04, label="Length scale (pixels)")
plt.axis("off")
plt.savefig("length-scale.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.linalg import cho_factor, cho_solve
from tqdm import tqdm

def reconstruct_region(
    region_id,
    blurred_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
):
    target_mask = region_mask == region_id
    if not target_mask.any():
        return None, None, None

    target_coords = np.column_stack(np.where(target_mask))
    image_shape = region_mask.shape
    n_target = len(target_coords)

    dilation_radius = int(np.ceil(4 * blur_sigma))
    latent_coords, latent_regions, obs_mask = get_neighborhood_pixels(
        target_mask, region_mask, dilation_radius
    )

    n_latent = len(latent_coords)

    obs_values = blurred_residual[obs_mask]
    valid_obs = ~np.isnan(obs_values) & (obs_values != 0)

    obs_coords_valid = latent_coords[valid_obs]
    obs_values_valid = obs_values[valid_obs]
    obs_regions_valid = latent_regions[valid_obs]
    n_obs = len(obs_coords_valid)

    if n_obs == 0:
        return None, None, None

    K_latent = build_block_diagonal_covariance(
        latent_coords, latent_regions, region_parameters
    )

    blur_offsets = build_blur_weights(blur_sigma)

    W_obs = np.zeros((n_obs, n_latent))
    for obs_idx in range(n_obs):
        obs_coord = obs_coords_valid[obs_idx]
        latent_idx = np.where((latent_coords == obs_coord).all(axis=1))[0][0]
        W_obs[obs_idx, :] = build_blur_matrix_row(
            latent_idx, latent_coords, latent_coords, blur_offsets, image_shape
        )

    K_obs = W_obs @ K_latent @ W_obs.T
    K_obs[np.diag_indices(n_obs)] += noise_variance

    target_indices = []
    for tc in target_coords:
        matches = np.where((latent_coords == tc).all(axis=1))[0]
        if len(matches) > 0:
            target_indices.append(matches[0])
        else:
            target_indices.append(-1)

    target_indices = np.array(target_indices)
    valid_target = target_indices >= 0

    if not valid_target.any():
        return None, None, None

    K_cross = np.zeros((n_target, n_obs))
    for i, tidx in enumerate(target_indices):
        if tidx >= 0:
            K_cross[i, :] = K_latent[tidx, :] @ W_obs.T

    target_var, _ = region_parameters[region_id]

    try:
        L, _ = cho_factor(K_obs)
        alpha = cho_solve((L, False), obs_values_valid)

        posterior_mean = np.full(n_target, np.nan)
        posterior_mean[valid_target] = K_cross[valid_target] @ alpha

        v = cho_solve((L, False), K_cross[valid_target].T)
        posterior_var = target_var - np.sum(K_cross[valid_target] * v.T, axis=1)
        posterior_var = np.maximum(posterior_var, 1e-10)

        posterior_std = np.full(n_target, np.nan)
        posterior_std[valid_target] = np.sqrt(posterior_var)

        return posterior_mean, posterior_std, target_coords

    except np.linalg.LinAlgError:
        print(f"LinAlgError in region {region_id}")
        return None, None, None


def compute_reconstruction(
    blurred_residual,
    region_mask,
    region_parameters,
    blur_sigma,
    noise_variance,
    max_region_size=5000,
):
    H, W = region_mask.shape
    posterior_mean = np.zeros((H, W))
    posterior_std = np.full((H, W), np.nan)

    regions = np.unique(region_mask)
    regions = regions[regions != 0]

    print(f"\nComputing GP reconstruction for {len(regions)} regions...")
    for region_id in tqdm(regions):
        n_pixels = np.sum(region_mask == region_id)
        if n_pixels == 0 or n_pixels > max_region_size:
            continue

        mean, std, coords = reconstruct_region(
            region_id,
            blurred_residual,
            region_mask,
            region_parameters,
            blur_sigma,
            noise_variance,
        )

        if mean is not None and coords is not None:
            posterior_mean[coords[:, 0], coords[:, 1]] = mean
            posterior_std[coords[:, 0], coords[:, 1]] = std

    return posterior_mean, posterior_std


original_residual = landsat_res_list[1]


blur_sigma = 0.97 * 7/3
blurred_residual = gaussian_filter(
    original_residual, sigma=blur_sigma, mode="constant", cval=0.0
)


reconstructed, uncertainties = compute_reconstruction(
    blurred_residual,
    mask_sub,
    region_parameters,
    blur_sigma=blur_sigma,
    noise_variance=1e-4,
    max_region_size=5000,
)


valid_vals = original_residual[~np.isnan(original_residual) & (original_residual != 0)]
if len(valid_vals) > 0:
    vmax = max(abs(np.percentile(valid_vals, 5)), abs(np.percentile(valid_vals, 95)))
    vmin = -vmax

    original_masked = original_residual.copy()
    original_masked[original_residual == 0] = np.nan

    plt.figure(figsize=(6, 5))
    im1 = plt.imshow(original_masked, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im1, fraction=0.046, pad=0.04, label="Residual")
    plt.axis("off")
    plt.savefig(
        "original-residual-t114.png", dpi=200, bbox_inches="tight", facecolor="white"
    )
    plt.show()

    blurred_masked = blurred_residual.copy()
    blurred_masked[blurred_residual == 0] = np.nan

    plt.figure(figsize=(6, 5))
    im2 = plt.imshow(blurred_masked, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im2, fraction=0.046, pad=0.04, label="Blurred residual")
    plt.axis("off")
    plt.savefig(
        "blurred-residual-t114.png", dpi=200, bbox_inches="tight", facecolor="white"
    )
    plt.show()

    reconstructed_masked = reconstructed.copy()
    reconstructed_masked[reconstructed == 0] = np.nan

    plt.figure(figsize=(6, 5))
    im3 = plt.imshow(reconstructed_masked, cmap="coolwarm", vmin=vmin, vmax=vmax)
    plt.colorbar(im3, fraction=0.046, pad=0.04, label="Prediction")
    plt.axis("off")
    plt.savefig(
        "reconstructed-residual-t114.png",
        dpi=200,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()

valid_std = uncertainties[~np.isnan(uncertainties)]
if len(valid_std) > 0:
    uncertainty_masked = uncertainties.copy() * 2
    uncertainty_masked[np.isnan(uncertainties) | (original_residual == 0)] = np.nan

    plt.figure(figsize=(6, 5))
    im4 = plt.imshow(uncertainty_masked, cmap="magma", vmin=0, vmax=0.2)
    plt.colorbar(im4, fraction=0.046, pad=0.04, label="Uncertainty")
    plt.axis("off")
    plt.savefig("uncertainty-t114.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()


reconstruction_error = np.abs(reconstructed - original_residual)
valid_mask = ~np.isnan(original_residual) & (original_residual != 0)

print(f"\n{'='*60}")
print("RECONSTRUCTION QUALITY METRICS")
print(f"{'='*60}")
print(f"Mean absolute error:      {np.mean(reconstruction_error[valid_mask]):.4f} °K")
print(f"Max absolute error:       {np.max(reconstruction_error[valid_mask]):.4f} °K")
print(
    f"RMSE:                     {np.sqrt(np.mean(reconstruction_error[valid_mask]**2)):.4f} °K"
)
print(f"Mean uncertainty (1σ):    {np.mean(uncertainties[valid_mask]):.4f} °K")
print(f"Mean uncertainty (2σ):    {2 * np.mean(uncertainties[valid_mask]):.4f} °K")
print(f"{'='*60}\n")

print(
    f"Posterior mean range: [{np.min(reconstructed[valid_mask]):.4f}, {np.max(reconstructed[valid_mask]):.4f}] °K"
)
print(f"Posterior mean std:   {np.std(reconstructed[valid_mask]):.4f} °K")
print(
    f"Uncertainty range:    [{np.min(uncertainties[valid_mask]):.4f}, {np.max(uncertainties[valid_mask]):.4f}] °K"
)


np.save("original_residual_t114.npy", original_residual)
np.save("blurred_residual_t114.npy", blurred_residual)
np.save("reconstructed_landsat_t114.npy", reconstructed)
np.save("uncertainties_landsat_t114.npy", uncertainties)

print("\nDone! Results saved.")

In [ ]:
reconstruction_residual = reconstructed_masked - original_masked
blurred_residual = blurred_masked - original_masked

vmax = max(abs(np.percentile(reconstruction_residual, 5)), abs(np.percentile(reconstruction_residual, 95)))
vmin = -vmax

plt.figure(figsize=(6, 5))
im1 = plt.imshow(reconstruction_residual, cmap="coolwarm", vmin=vmin, vmax=vmax)
plt.colorbar(im1, fraction=0.046, pad=0.04, label="Residual")
plt.axis("off")

reconstruction_residual = reconstructed_masked - original_masked
blurred_residual = blurred_masked - original_masked

# vmax = max(abs(np.percentile(blurred_residual, 5)), abs(np.percentile(blurred_residual, 95)))
# vmin = -vmax

plt.figure(figsize=(6, 5))
im1 = plt.imshow(blurred_residual, cmap="coolwarm", vmin=vmin, vmax=vmax)
plt.colorbar(im1, fraction=0.046, pad=0.04, label="Residual")
plt.axis("off")

In [ ]:
np.nanmean(np.abs(reconstruction_residual)), np.nanmean(np.abs(blurred_residual))